# 20 — Virtual cohort: plan (d)

**Scope. autoPET is mostly low-burden staging patients, not mCRPC therapy candidates: this virtual cohort does not represent the Pluvicto population (§8).**

**What a virtual patient is.** A real phantom (anatomy, real PET as network input) plus one draw of its kinetics: organ densities and nuisance from the **posterior** of its real PET, lesion density and flow from the joint draw, $\lambda_{rel}$ and the agent factors $\phi$ from the population calibrated in nb19. Not point estimates.

| Setting | Value |
|---|---|
| Training | 180 training phantoms, 10 draws per phantom, 5 scenarios, main arm |
| Test | 47 test phantoms, same settings |
| Control | $\phi=1$, its own calibrated $\lambda_{rel}$, same lesion median, 2 draws per phantom, scenario A |
| Therapy | 7.4 GBq; PSMA-617 mass measured in clinical doses (Wang 2025), per patient, about 45-80 nmol |
| Scenario A | washout independent of PET uptake |
| Scenario B | correlation $\rho$ of $\log\lambda_{rel}$ with log PET uptake, -0.6, -0.3, +0.3, +0.6 |
| Outputs | TIA per Monte Carlo source and per lesion for 177Lu, simulated 68Ga PET values, kinetic parameters, all parameters per virtual patient |

**The test split.** The test phantoms are fitted here for the first time, because a posterior draw needs the PET of its phantom. **The population is frozen:** everything comes from the training split, nothing is learned from the test fits, and no statistic of the test cohort is shown or used in this notebook, only counts and failures. So the test set stays a test set. Its PETs have now been read to build the targets, as any test set must be at evaluation.

**Dose is not here.** It will come from the Monte Carlo $D_c$ when the full run ends. Doses below are by S-value and are only checks against the calibration targets. This notebook does not touch the Monte Carlo run or its files.

**$\rho$ means:** positive $\rho$ gives faster release to higher uptake. Permutation of one pool of draws, so the marginals of $\lambda_{rel}$ are **identical** in every scenario. No source fixes $\rho$; the scenarios bracket it (nb12).

**TIA per source.** Kidneys, liver, spleen, glands, GI directly. Blood pool: blood concentration over the aorta mask. Spine: bone and marrow concentration over the spine mask. Rest of body: everything else in the body, so nothing is counted twice, lesions apart. Bladder: **voiding every 4 h, assumption**, TIA = 2 h × decayed activity reaching the urine. **Kinetic parameters:** $A(t)=A_0(e^{-l_w t}-e^{-l_u t})$, $l_w$ and $A_0$ from the slope over 24-168 h, $l_u$ so that the integral equals the TIA, $l_u=\infty$ when the slope alone already exceeds it.

**About this notebook.** Frozen virtual cohort: posterior draws per phantom, scenarios A and B, therapy TIA and kinetic parameters.
- **Inputs:** nb18, nb19 outputs.
- **Outputs:** `data/processed/psma/cohort/` (frozen, checksums in `cohort_manifest.csv`); do not regenerate.
- **Run time:** about 26 min.
- **Environment:** .venv.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import json
import multiprocessing as mp
import pickle
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import pbpk_calibration as C
import pbpk_cohort as Co
import pbpk_fit as F

warnings.filterwarnings("ignore", category=UserWarning)
PBPK = ROOT / "data/processed/psma/pbpk"
LIMIT = int(os.environ.get("NB20_LIMIT", 0))
OUT = ROOT / ("data/processed/psma/cohort" + ("_smoke" if LIMIT else ""))
OUT.mkdir(exist_ok=True)
N_PROCESSES = 3
SEED = 20261107
N_DRAWS, N_CONTROL = (10, 2) if not LIMIT else (3, 1)
RHOS = [-0.6, -0.3, 0.3, 0.6]
SCENARIOS = ["A"] + [f"B{r:+.1f}" for r in RHOS]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9, "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": "#b8b7ae"})
pd.set_option("display.width", 230, "display.max_columns", 40, "display.max_rows", 150)


def cached(name, build):
    path = OUT / f"{name}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    value = build()
    path.write_bytes(pickle.dumps(value))
    return value


population = pickle.loads((PBPK / "nb19_cohort_population_v2.pkl").read_bytes())
COHORT_HYPER = lambda median: replace(F.LesionHyper(mu=median), tau_between=F.LesionHyper().tau_between * population["tau_scale"], tau_within=F.LesionHyper().tau_within * population["tau_scale"])
split = json.loads((ROOT / "data/processed/psma/psma_68ga_split.json").read_text())
TRAIN, TEST = split["train_studies"], split["test_studies"]
assert not set(TRAIN) & set(TEST)
if LIMIT:
    TRAIN, TEST = TRAIN[:LIMIT], TEST[:2]
SETTINGS = population["settings"]
print(len(TRAIN), "training phantoms,", len(TEST), "test phantoms")
print("Lesion median", population["hyper_median"], "| SD scale", population["tau_scale"], "| therapy amount measured per patient")
print("tau", population["tau"], "| kidneys", SETTINGS["priors"]["kidneys"], "| datum error", SETTINGS["extra_sigma"])
print("phi SD", population["main"]["phi_sd"], "| lambda_rel SD", population["rel_sd"], "| lesion epistemic SD", round(population["lesion_epistemic_sd"], 3))
display(pd.DataFrame(dict(main=np.exp(population["main"]["theta"]), control=np.exp(population["control"]["theta"])), index=C.AGENT_NAMES).map(lambda v: float(f"{v:.3g}")))


def safe(function, job):
    try:
        return function(job)
    except (ValueError, RuntimeError, np.linalg.LinAlgError) as error:
        return dict(error=repr(error)[:200], job=str(job)[:80])


def bundle_job(study):
    return safe(Co.stage_bundle, (study, SETTINGS, COHORT_HYPER(population["hyper_median"]), 30))


def bundles_build(ids):
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(bundle_job, ids, chunksize=2)


train_bundles = cached("bundles_train", lambda: bundles_build(TRAIN))
test_bundles = cached("bundles_test", lambda: bundles_build(TEST))
for name, group in [("train", train_bundles), ("test", test_bundles)]:
    print(name, "bundles ok", sum("error" not in b for b in group), "of", len(group), *[b["error"] for b in group if "error" in b][:2])
train_ok = [b for b in train_bundles if "error" not in b]
test_ok = [b for b in test_bundles if "error" not in b]
print("Importance ESS of 300, training: median", round(np.median([b["ess"] for b in train_ok]), 1), "| p10", round(np.quantile([b["ess"] for b in train_ok], 0.1), 1))

## 1. Draws: parameters and simulated PET

Per virtual patient, seed from (cohort seed, phantom, draw, arm), so each is reproducible alone.

In [ ]:
ARMS = dict(main=dict(n=N_DRAWS, population=population["main"], median=population["hyper_median"]), control=dict(n=N_CONTROL, population=population["control"], median=population["hyper_median_control"]))
STRUCTURES = ("kidneys", "liver", "salivary")


def make_virtual(job):
    bundle, draw, arm = job
    config = ARMS[arm]
    pop = config["population"]
    rng = np.random.default_rng([SEED, F.hash_id(bundle["study_id"]), draw, 0 if arm == "main" else 1])
    g = bundle["geometry"]
    n = len(g.lesions_ml)
    values = Co.draw_values(bundle, draw, SETTINGS["priors"])
    theta = pop["theta"]
    hyper = COHORT_HYPER(config["median"])
    r_lesions, f_lesions = F.draw_lesions(bundle["likelihood"], hyper, rng) if n else (np.zeros(0), np.zeros(0))
    log_rel = dict(kidneys=theta[4], liver=theta[5], salivary=theta[6])
    rel = {k: float(np.exp(log_rel[k] + population["rel_sd"][k] * rng.normal())) for k in STRUCTURES}
    lesion_median = theta[7] + population["lesion_epistemic_sd"] * rng.normal()
    l_lesions = np.exp(lesion_median + population["rel_sd"]["lesions"] * rng.normal(size=n))
    factors = {k: float(np.exp(m + pop["phi_sd"][k] * rng.normal())) for k, m in zip(C.AGENT_ORGANS, theta[:4])}
    pet = Co.simulate_pet(g, values, r_lesions, f_lesions, l_lesions, rel)
    therapy_nmol = C.measured_amount((F.hash_id(bundle["study_id"]) + 104729 * draw) % 2**31)
    return dict(therapy_nmol=therapy_nmol, tumour_sites_nmol=float(np.sum(r_lesions * g.lesions_ml) / 1000), study_id=bundle["study_id"], subject=bundle["subject"], scanner=bundle["scanner"], draw=draw, arm=arm, values=values, r_lesions=r_lesions, f_lesions=f_lesions, l_lesions=l_lesions, rel=rel, factors=factors, pet=pet, volumes=g.lesions_ml, organ_ml=dict(kidneys=g.kidneys_ml, liver=g.liver_ml, spleen=g.spleen_ml, salivary=g.salivary_ml or 1.0))


def virtual_job(job):
    return safe(make_virtual, job)


def virtual_build(group, arm):
    jobs = [(b, k, arm) for b in group for k in range(ARMS[arm]["n"])]
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(virtual_job, jobs, chunksize=4)


virtual = {}
for split_name, group in [("train", train_ok), ("test", test_ok)]:
    for arm in ARMS:
        output = cached(f"virtual_{split_name}_{arm}", lambda: virtual_build(group, arm))
        failed = [x for x in output if "error" in x]
        print(split_name, arm, "virtual patients ok", len(output) - len(failed), "of", len(output), *[x["error"] for x in failed][:1])
        virtual[split_name, arm] = [x for x in output if "error" not in x]

## 2. Scenario B: one pool of $\lambda_{rel}$, assigned by rank

For each structure (kidneys; liver with spleen; glands; lesions, pooled over all lesions) and within each split: the normal score $u$ of log PET concentration, a latent $z=\rho u+\sqrt{1-\rho^2}\,\varepsilon$, and the sorted pool of $\log\lambda_{rel}$ assigned by the rank of $z$. **The pool is the set of scenario A values, so marginals are identical by construction,** checked below. The test split has its own pool and its own ranks.

In [ ]:
def normal_scores(x):
    return stats.norm.ppf((stats.rankdata(x) - 0.5) / len(x))


def assign(pool, uptake, rho, rng):
    z = rho * normal_scores(uptake) + np.sqrt(1 - rho**2) * rng.normal(size=len(pool))
    out = np.zeros(len(pool))
    out[np.argsort(z)] = np.sort(pool)
    return out


def scenario_table(items, seed_tag):
    out = {"A": dict(organs=[dict(x["rel"]) for x in items], lesions=[x["l_lesions"].copy() for x in items])}
    uptake = {k: np.log([x["pet"][k] / x["organ_ml"][k] for x in items]) for k in STRUCTURES}
    pools = {k: np.log([x["rel"][k] for x in items]) for k in STRUCTURES}
    lesion_counts = [len(x["l_lesions"]) for x in items]
    lesion_pool = np.log(np.concatenate([x["l_lesions"] for x in items])) if sum(lesion_counts) else np.zeros(0)
    lesion_uptake = np.log(np.concatenate([x["pet"]["lesions"] / x["volumes"] for x in items if len(x["l_lesions"])])) if sum(lesion_counts) else np.zeros(0)
    for rho in RHOS:
        rng = np.random.default_rng([SEED, seed_tag, int(round((rho + 1) * 10))])
        assigned = {k: np.exp(assign(pools[k], uptake[k], rho, rng)) for k in STRUCTURES}
        lesions = np.exp(assign(lesion_pool, lesion_uptake, rho, rng))
        parts = np.split(lesions, np.cumsum(lesion_counts)[:-1])
        out[f"B{rho:+.1f}"] = dict(organs=[{k: float(assigned[k][i]) for k in STRUCTURES} for i in range(len(items))], lesions=parts)
    return out, uptake, lesion_uptake


tables = {}
for split_name in ("train", "test"):
    tables[split_name] = scenario_table(virtual[split_name, "main"], 1 if split_name == "train" else 2)
rows = []
table, uptake, lesion_uptake = tables["train"]
for name in SCENARIOS:
    row = dict(scenario=name)
    for k in STRUCTURES:
        values = np.array([x[k] for x in table[name]["organs"]])
        row[f"spearman_{k}"] = stats.spearmanr(np.log(values), uptake[k])[0]
        row[f"marginal_same_{k}"] = bool(np.array_equal(np.sort(values), np.sort([x[k] for x in table["A"]["organs"]])) or np.allclose(np.sort(values), np.sort([x[k] for x in table["A"]["organs"]]), rtol=1e-10))
    lesions = np.concatenate(table[name]["lesions"])
    row["spearman_lesions"] = stats.spearmanr(np.log(lesions), lesion_uptake)[0]
    row["marginal_same_lesions"] = bool(np.allclose(np.sort(lesions), np.sort(np.concatenate(table["A"]["lesions"])), rtol=1e-10))
    rows.append(row)
scenario_check = pd.DataFrame(rows).set_index("scenario")
scenario_check.to_csv(OUT / "check_scenarios.csv")
display(scenario_check.round(3))
for name in SCENARIOS:
    for k in STRUCTURES:
        a, b = np.sort([x[k] for x in tables["test"][0][name]["organs"]]), np.sort([x[k] for x in tables["test"][0]["A"]["organs"]])
        assert np.allclose(a, b, rtol=1e-10)
print("Marginals identical in every scenario, also on the test split (assertion only, no statistic shown).")

## 3. Therapy: TIA, kinetic parameters

Full topology, lesions as decided in nb12 (satellites up to 1 L of TLV, coupled above). Every virtual patient in every scenario.

In [ ]:
SPINE = {}
for b in train_ok + test_ok:
    SPINE[b["study_id"]] = Co._spine_ml(b["study_id"])
GEOMETRY = {b["study_id"]: b["geometry"] for b in train_ok + test_ok}


def therapy_job(job):
    x, scenario, rel, lesions = job

    def go(_):
        g = GEOMETRY[x["study_id"]]
        spine, blood = SPINE[x["study_id"]]
        out = Co.simulate_therapy(g, x["values"], x["r_lesions"], x["f_lesions"], lesions, rel, x["factors"], spine, blood, nmol=x["therapy_nmol"])
        return dict(study_id=x["study_id"], draw=x["draw"], arm=x["arm"], scenario=scenario, **out)
    return safe(go, None)


def therapy_build(items, table, names):
    jobs = [(x, name, table[name]["organs"][i], table[name]["lesions"][i]) for name in names for i, x in enumerate(items)]
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(therapy_job, jobs, chunksize=6)


therapy = {}
for split_name in ("train", "test"):
    items = virtual[split_name, "main"]
    output = cached(f"therapy_{split_name}_main", lambda: therapy_build(items, tables[split_name][0], SCENARIOS))
    failed = [x for x in output if "error" in x]
    print(split_name, "main ok", len(output) - len(failed), "of", len(output), *[x["error"] for x in failed][:1])
    therapy[split_name, "main"] = [x for x in output if "error" not in x]
    items = virtual[split_name, "control"]
    control_table = {"A": dict(organs=[x["rel"] for x in items], lesions=[x["l_lesions"] for x in items])}
    output = cached(f"therapy_{split_name}_control", lambda: therapy_build(items, control_table, ["A"]))
    failed = [x for x in output if "error" in x]
    print(split_name, "control ok", len(output) - len(failed), "of", len(output), *[x["error"] for x in failed][:1])
    therapy[split_name, "control"] = [x for x in output if "error" not in x]

## 4. Tables per virtual patient

`virtual_id` = phantom and draw. One parquet file per table and split in `data/processed/psma/cohort/`: `patients` (densities, nuisance, $\phi$, seed), `lesions` ($[R]$, flow, volume), `lambda_rel` (per scenario), `lesion_lambda_rel` (per scenario), `pet` (per compartment and lesion), `tia` (per source and lesion, per scenario, with $A_0$, $l_w$, $l_u$), `checks` (S-value doses and half-lives, not outputs).

In [ ]:
def build_tables(split_name):
    patients, lesions, pet_rows, lambda_rows, lesion_lambda_rows, tia_rows, check_rows = [], [], [], [], [], [], []
    for arm in ARMS:
        items = virtual[split_name, arm]
        table = tables[split_name][0] if arm == "main" else {"A": dict(organs=[x["rel"] for x in items], lesions=[x["l_lesions"] for x in items])}
        for i, x in enumerate(items):
            vid = f"{x['study_id']}_d{x['draw']:02d}"
            key = dict(split=split_name, arm=arm, virtual_id=vid)
            patients.append({**key, "study_id": x["study_id"], "subject": x["subject"], "scanner": x["scanner"], "draw": x["draw"], "seed": SEED, **{f"r_{k}": v for k, v in x["values"]["r_den"].items()}, **{k: x["values"][k] for k in ("f_sal", "gfr", "hematocrit", "amount_nmol")}, **{f"phi_{k}": v for k, v in x["factors"].items()}, "tlv_ml": float(x["volumes"].sum()), "therapy_nmol": x["therapy_nmol"], "tumour_sites_nmol": x["tumour_sites_nmol"]})
            for j in range(len(x["volumes"])):
                lesions.append({**key, "lesion": j, "volume_ml": x["volumes"][j], "r_den": x["r_lesions"][j], "flow": x["f_lesions"][j], "pet_pct_ia": x["pet"]["lesions"][j]})
            for name in ("kidneys", "liver", "spleen", "salivary", "gi", "urine"):
                pet_rows.append({**key, "compartment": name, "pct_ia": x["pet"][name]})
            pet_rows.append({**key, "compartment": "blood_conc_per_ml", "pct_ia": x["pet"]["blood_conc"]})
            for name in table:
                lambda_rows.append({**key, "scenario": name, **{f"rel_{k}": v for k, v in table[name]["organs"][i].items()}})
                for j, value in enumerate(table[name]["lesions"][i]):
                    lesion_lambda_rows.append({**key, "scenario": name, "lesion": j, "rel": value})
        for t in therapy[split_name, arm]:
            vid = f"{t['study_id']}_d{t['draw']:02d}"
            key = dict(split=split_name, arm=arm, virtual_id=vid, scenario=t["scenario"])
            for name, value in t["tia"].items():
                tia_rows.append({**key, "compartment": name, "tiac_h": value, **t["kinetics"][name]})
            for j, value in enumerate(t["lesion_tia"]):
                tia_rows.append({**key, "compartment": f"lesion_{j}", "tiac_h": value, **t["lesion_kinetics"][j]})
            check_rows.append({**key, **t["check"]})
    for name, rows_ in [("patients", patients), ("lesions", lesions), ("pet", pet_rows), ("lambda_rel", lambda_rows), ("lesion_lambda_rel", lesion_lambda_rows), ("tia", tia_rows), ("checks", check_rows)]:
        frame_ = pd.DataFrame(rows_)
        if name == "tia":
            frame_["tiac_late_h"] = frame_.a0 / frame_.lw
            frame_["tiac_early_h"] = frame_.tiac_h - frame_.tiac_late_h
        frame_.to_parquet(OUT / f"{name}_{split_name}.parquet", index=False)
    return {name: len(rows_) for name, rows_ in [("patients", patients), ("lesions", lesions), ("pet", pet_rows), ("tia", tia_rows), ("checks", check_rows)]}


for split_name in ("train", "test"):
    print(split_name, build_tables(split_name))
print("Size on disk, MB:", round(sum(p.stat().st_size for p in OUT.glob("*.parquet")) / 1e6, 1))

## 5. Checks on the generated training cohort

Posterior draws, not point estimates: do doses, half-lives and spreads still match? Mean dose and median half-life over virtual patients; log SD between phantoms, as a real cohort would see it (one draw per phantom), and over all draws.

In [ ]:
checks = pd.read_parquet(OUT / "checks_train.parquet")
main_a, control_a = checks[(checks.arm == "main") & (checks.scenario == "A")], checks[(checks.arm == "control") & (checks.scenario == "A")]
targets = population["targets"]
rows = []
for name in targets.index:
    if name not in main_a:
        continue
    pick = (lambda d: d[name].median()) if name.startswith("teff") else (lambda d: d[name].mean())
    rows.append(dict(target=name, published=targets.loc[name, "value"], tolerance=targets.loc[name, "tolerance"], used=targets.loc[name, "used"], main=pick(main_a), main_ratio=pick(main_a) / targets.loc[name, "value"], control=pick(control_a), control_ratio=pick(control_a) / targets.loc[name, "value"]))
against = pd.DataFrame(rows).set_index("target")
against.to_csv(OUT / "check_against_targets.csv")
display(against.round(3))
one_draw = main_a[main_a.virtual_id.str.endswith("_d00")]
spread_check = pd.DataFrame({name: dict(target=population["spread_targets"][name], all_draws=np.log(main_a[name].dropna()).std(), one_draw_per_phantom=np.log(one_draw[name].dropna()).std(), control=np.log(control_a[name].dropna()).std()) for name in population["spread_targets"]}).T
spread_check.to_csv(OUT / "check_spreads.csv")
display(spread_check.round(3))
patients = pd.read_parquet(OUT / "patients_train.parquet")
main_patients = patients[patients.arm == "main"].set_index("virtual_id")
frame = main_a.set_index("virtual_id").join(main_patients[["tlv_ml", "study_id"]])
display(frame.groupby(pd.cut(frame.tlv_ml, [-1, 10, 100, 300, 1e5]), observed=True).agg(n=("teff_whole_body", "size"), teff_whole_body=("teff_whole_body", "median"), dose_kidneys=("dose_kidneys", "mean"), dose_salivary=("dose_salivary", "mean"), dose_lesions=("dose_lesions", "mean")).round(3))
by_scenario = checks[checks.arm == "main"].groupby("scenario")[[c for c in checks if c.startswith("dose")]].mean().join(checks[checks.arm == "main"].groupby("scenario")[[c for c in checks if c.startswith("teff")]].median())
by_scenario.to_csv(OUT / "check_by_scenario.csv")
display(by_scenario.round(3))
pet = pd.read_parquet(OUT / "pet_train.parquet")
pet_main = pet[pet.arm == "main"].pivot(index="virtual_id", columns="compartment", values="pct_ia")
pet_main["study_id"] = main_patients.loc[pet_main.index, "study_id"]
real = pd.DataFrame([dict(study_id=b["study_id"], kidneys=b["obs"]["kidneys"], liver=b["obs"]["liver"] * b["geometry"].liver_ml, spleen=b["obs"]["spleen"] * b["geometry"].spleen_ml, salivary=b["obs"].get("salivary", np.nan) * (b["geometry"].salivary_ml or 1.0), blood=b["obs"]["blood"]) for b in train_ok]).set_index("study_id")
virtual_means = pet_main.groupby("study_id")[["kidneys", "liver", "spleen", "salivary", "blood_conc_per_ml"]].median().rename(columns=dict(blood_conc_per_ml="blood"))
predictive = pd.DataFrame({k: dict(median_log_ratio=np.log(virtual_means[k] / real[k]).median(), sd_log_ratio=np.log(virtual_means[k] / real[k]).std(), r_log=np.corrcoef(*np.log(pd.concat([virtual_means[k], real[k]], axis=1).dropna().to_numpy().T))[0, 1]) for k in virtual_means}).T
predictive.to_csv(OUT / "check_pet_predictive.csv")
print("Simulated PET of the draws against the real datum, per phantom")
display(predictive.round(3))
burden_frame = frame.join(main_patients[["therapy_nmol", "tumour_sites_nmol"]])
burden_high = burden_frame[burden_frame.tlv_ml >= 100]
LESION_HIGH = population["lesion_high_target"]
final_burden = dict(rho_kidneys_high=stats.spearmanr(burden_high.dose_kidneys, burden_high.tlv_ml)[0], rho_glands_high=stats.spearmanr(burden_high.dose_salivary, burden_high.tlv_ml)[0], rho_kidneys_all=stats.spearmanr(burden_frame.dose_kidneys, burden_frame.tlv_ml)[0], rho_glands_all=stats.spearmanr(burden_frame.dose_salivary, burden_frame.tlv_ml)[0], lesion_high=burden_high.dose_lesions.mean(), lesion_high_target=LESION_HIGH, lesion_spread=np.log(burden_frame.dose_lesions.dropna()).std(), lesion_spread_target=population["spread_targets"]["dose_lesions"], sites_high=burden_high.tumour_sites_nmol.median(), amount_median=burden_frame.therapy_nmol.median(), share_sites_over_amount=(burden_high.tumour_sites_nmol > burden_high.therapy_nmol).mean())
pd.Series(final_burden).to_csv(OUT / "check_burden_final.csv")
print("Burden targets on the final cohort; Violet: -0.43, -0.41 at mCRPC burden")
display(pd.Series(final_burden).round(3).to_frame("final cohort").T)
burden_rank = pd.DataFrame({name: dict(spearman_with_tlv=stats.spearmanr(frame.dropna(subset=[name])[name], frame.dropna(subset=[name]).tlv_ml)[0]) for name in ["dose_kidneys", "dose_liver", "dose_spleen", "dose_salivary", "dose_lesions"]}).T
burden_rank.to_csv(OUT / "check_dose_vs_burden.csv")
print("Dose against TLV. Violet 2019: tumour volume with parotid dose r = -0.41, with kidney dose r = -0.43")
display(burden_rank.round(3))
tia = pd.read_parquet(OUT / "tia_train.parquet")
tia = tia[(tia.arm == "main") & (tia.scenario == "A")]
tia["group"] = np.where(tia.compartment.str.startswith("lesion"), "lesions", tia.compartment)
tia["early_share"] = tia.tiac_early_h / tia.tiac_h
shape = tia.groupby("group").agg(tiac_h=("tiac_h", "median"), t_late_h=("lw", lambda v: np.log(2) / v.median()), early_share=("early_share", "median"), no_uptake_phase=("lu", lambda v: np.isinf(v).mean()))
shape.to_csv(OUT / "check_kinetic_shape.csv")
print("Kinetic parameters. no_uptake_phase: share of curves where the 24-168 h slope alone does not reach the TIA, so lu is infinite and tiac_early_h is positive")
display(shape.round(3))
fig, axes = plt.subplots(1, 4, figsize=(10, 2.6), constrained_layout=True)
for ax, name, label in zip(axes, ["dose_kidneys", "dose_liver", "dose_salivary", "dose_lesions"], ["Kidneys", "Liver", "Glands", "Lesions"]):
    v = main_a[name].dropna()
    ax.hist(np.log10(v), bins=30, color=COLORS[0], alpha=0.8)
    ax.axvline(np.log10(targets.loc[name, "value"]), color=INK, lw=1.2)
    ax.set(xlabel=f"{label}, log10 Gy/GBq")
axes[0].set_ylabel("Virtual patients")
fig.savefig(OUT / "check_doses.png", dpi=160)

## 6. Report: the frozen cohort

**Generated, no failures:** training 1800 virtual patients (180 phantoms × 10) × 5 scenarios, control 360; test 470 × 5, control 94. Importance ESS of 300: median 161, p10 91. Seed 20261107, the same as the first cohort. All parameters per virtual patient, therapy amount and tumour sites included. Tables in `data/processed/psma/cohort/`; checksums in `data/processed/psma/cohort_manifest.csv`. The first cohort stays in `cohort_v1/`. **The Monte Carlo run was not touched, and no dose from it is here.**

**Test split.** Fitted with the population frozen, as stated at the top. Only counts and failures are shown.

**Population (nb19 §17):** lesion median 800 nmol/L, SD scale 1.15, therapy amount measured per patient (median 60 nmol, 5-95% 45-80). $\phi$ kidneys 0.28, glands 0.19, liver 1.77, spleen 0.71. $\lambda_{rel}$ kidneys 1.88e-4, liver and spleen 3.10e-4, glands 2.73e-4, lesions 1.05e-4 /min. Control: $\lambda_{rel}$ 2.33e-4, 2.93e-4, 2.83e-4, 1.02e-4.

**On the generated training cohort, posterior draws, scenario A.**

| | Kidneys | Liver | Spleen | Glands | Lesions | Whole body |
|---|---|---|---|---|---|---|
| Mean dose / target, main | 1.15 | 1.00 | 1.01 | 1.03 | 1.28 (all), **1.03 at high burden** | |
| Mean dose / target, control | 3.46 | 0.59 | 1.38 | 4.06 | 1.32 | |
| Half-life / target, main | 1.24 | 1.09 | | 1.00 | 0.96 (not a target) | 0.84 |
| Log SD of dose, target | 0.34 | 0.44 | 0.25 | 0.46 | 0.78 | |
| Log SD of dose, cohort | 0.39 | 0.45 | 0.49 | 0.50 | **0.83** | |

Lesion half-life comes to 66 h against 69 h without being a target, which the first cohort did not do (86 h).

**Scenarios.** Marginals of $\lambda_{rel}$ identical, by permutation, checked on both splits. Realised Spearman within 0.03 of $\rho$ on all structures, except liver at -0.6 (-0.63), and at most 0.03 in scenario A. Organ means hardly change across scenarios; the median lesion half-life moves from 59 to 74 h, because it is taken over patients and lesions at or above 0.5 mL.

**Simulated PET of the draws against the real datum, per phantom:** liver, spleen, glands within 1.5%, r 0.95-1.00; blood -4%, r 0.73. **Kidneys -11%, SD 0.31, r 0.69,** by the declared datum error: the real PET input carries kidney variation that the target does not follow.

## 7. Tumour-burden dependence: the round on the first cohort

This section reads the first cohort, kept in `data/processed/psma/cohort_v1/` (lesion median 181, 100 nmol), and its cached results.

**Why.** §6: no organ dose falls with TLV, against Violet 2019, and lesions are low at high burden.

**Diagnosis.** (a) Fitted $[R_K]$ and $f_{SAL}$ against TLV: does the PET fit compensate the sink? (b) Kidney and gland concentration across TLV tertiles: real PET, the PET of the draws, and the model at one population density for all patients, which is the sink of the model alone. (c) Total tumour binding sites $\sum R_j V_j$ against 100 nmol. (d) The same therapy with no per-patient kidney and gland information: population density for every patient.

**Targets added.** Spearman of dose with TLV, Violet 2019: kidneys -0.43, glands (parotid) -0.41, n = 30, so a 95% interval of about ±0.3. Lesion Gy/GBq at high burden: geometric mean of Ells 2026 whole tumour burden 1.84, Rosar 2021 bone 1.68, Violet 2019 whole-body tumour 1.48 (mCRPC). Lesion dose spread 0.78 (§5). Parameters: lesion median, a scale on its SDs, and the therapy amount, 50-200 nmol (assumed plausible range, no source fixes it for these sources).

**Criterion, fixed before the grid.** It closes if, at one setting, on phantoms with TLV at or above 100 mL (mCRPC-like, as Violet's cohort is): Spearman kidneys and glands both inside -0.43 ± 0.3 and -0.41 ± 0.3; lesion dose at high burden within its tolerance of 0.3 in log (assumed, about the spread of those three); lesion dose spread within 0.2 of the target. **If it closes, the cohort is regenerated with the same seeds; if not, the current cohort stays.**

In [ ]:
from scipy.stats import spearmanr

V1 = ROOT / "data/processed/psma/cohort_v1"
population_v1 = pickle.loads((PBPK / "nb19_cohort_population.pkl").read_bytes())
SETTINGS_V1 = population_v1["settings"]
v1_ok = [b for b in pickle.loads((V1 / "bundles_train.pkl").read_bytes()) if "error" not in b]


def cached_v1(name, build):
    path = V1 / f"{name}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    value = build()
    path.write_bytes(pickle.dumps(value))
    return value



bundle_frame = pd.DataFrame([dict(study_id=b["study_id"], tlv_ml=float(b["geometry"].lesions_ml.sum()), r_kidneys=float(np.exp(b["draws"][:, b["r_keys"].index("kidneys")]).mean()), map_kidneys=b["map_r"]["kidneys"], f_sal=b["map_nuisance"].f_sal, gfr=b["map_nuisance"].gfr, real_kidneys=b["obs"]["kidneys"] / b["geometry"].kidneys_ml, real_salivary=b["obs"].get("salivary", np.nan), real_liver=b["obs"]["liver"], real_spleen=b["obs"]["spleen"]) for b in v1_ok]).set_index("study_id")
rank = pd.DataFrame({k: dict(spearman_with_tlv=spearmanr(bundle_frame.tlv_ml, bundle_frame[k], nan_policy="omit")[0], p=spearmanr(bundle_frame.tlv_ml, bundle_frame[k], nan_policy="omit")[1]) for k in ["r_kidneys", "map_kidneys", "f_sal", "gfr", "real_kidneys", "real_salivary", "real_liver", "real_spleen"]}).T
rank.to_csv(V1 / "burden_fit_rank.csv")
display(rank.round(3))
main_theta = population_v1["main"]["theta"]
PRIORS_FROZEN = SETTINGS_V1["priors"]


def population_pet_job(bundle):
    g = bundle["geometry"]
    n = len(g.lesions_ml)
    rng = np.random.default_rng([SEED, F.hash_id(bundle["study_id"]), 500])
    r_les, f_les = F.draw_lesions(bundle["likelihood"], F.LesionHyper(mu=population_v1["hyper_median"]), rng) if n else (np.zeros(0), np.zeros(0))
    r_den = dict(kidneys=PRIORS_FROZEN["kidneys"][0], liver=PRIORS_FROZEN["liver"][0], spleen=PRIORS_FROZEN["spleen"][0], salivary=F.PRIORS["salivary"][0])
    draw = dict(r_den=r_den, f_sal=PRIORS_FROZEN["f_sal"][0], gfr=F.NUISANCE_PRIOR["gfr"][0], hematocrit=F.NUISANCE_PRIOR["hematocrit"][0], amount_nmol=F.NUISANCE_PRIOR["amount_nmol"][0])
    pet = Co.simulate_pet(g, draw, r_les, f_les, np.full(n, np.exp(main_theta[7])), dict(kidneys=np.exp(main_theta[4]), liver=np.exp(main_theta[5]), salivary=np.exp(main_theta[6])))
    return dict(study_id=bundle["study_id"], model_kidneys=pet["kidneys"] / g.kidneys_ml, model_salivary=pet["salivary"] / (g.salivary_ml or 1.0))


def population_pet_build():
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        return workers.map(population_pet_job, v1_ok, chunksize=4)


model_pet = pd.DataFrame(cached_v1("burden_population_pet", population_pet_build)).set_index("study_id")
pet_main_all = pd.read_parquet(V1 / "pet_train.parquet")
pet_main_all = pet_main_all[(pet_main_all.arm == "main") & pet_main_all.compartment.isin(["kidneys", "salivary"])].pivot(index="virtual_id", columns="compartment", values="pct_ia")
pet_main_all["study_id"] = [v.rsplit("_d", 1)[0] for v in pet_main_all.index]
volumes = pd.Series({b["study_id"]: b["geometry"].kidneys_ml for b in v1_ok})
glands = pd.Series({b["study_id"]: b["geometry"].salivary_ml or 1.0 for b in v1_ok})
draws_pet = pet_main_all.groupby("study_id")[["kidneys", "salivary"]].median()
draws_pet = pd.DataFrame(dict(draws_kidneys=draws_pet.kidneys / volumes, draws_salivary=draws_pet.salivary / glands))
slopes = bundle_frame.join(model_pet).join(draws_pet)
slopes["tertile"] = pd.qcut(slopes.tlv_ml.rank(method="first"), 3, labels=["low", "middle", "high"])
tertiles = slopes.groupby("tertile", observed=True)[["tlv_ml", "real_kidneys", "draws_kidneys", "model_kidneys", "real_salivary", "draws_salivary", "model_salivary", "r_kidneys", "f_sal"]].median()
tertiles.to_csv(V1 / "burden_tertiles.csv")
display(tertiles.map(lambda v: float(f"{v:.3g}")))
display((tertiles.loc["high"] / tertiles.loc["low"]).round(3).to_frame("high over low").T)

In [ ]:
GRID_MEDIAN = [100.0, 181.0, 400.0, 800.0, 1600.0, 3000.0]
GRID_TAU = [0.5, 1.0, 1.3]
GRID_NMOL = [50.0, 100.0, 200.0]
phi_list = [population_v1["main"]["phi_sd"][k] for k in C.AGENT_ORGANS]
rel_list = [population_v1["rel_sd"][k] for k in ("kidneys", "liver", "salivary", "lesions")]
BASES = Co.calibration_bases(v1_ok, SETTINGS_V1["priors"], per_patient=2)
TLV = np.array([b.geometry.lesions_ml.sum() for b in BASES])
HIGH = TLV >= 100.0
LESION_HIGH_TARGET = float(np.exp(np.mean(np.log([1.84, 1.68, 11.55 / 7.8]))))
print(len(BASES), "simulations per setting;", int(HIGH.sum()), "at or above 100 mL; lesion target at high burden", round(LESION_HIGH_TARGET, 2))


def grid_job(job):
    index, median, tau, nmol, flat = job
    base = BASES[index]
    if flat:
        base = replace(base, r_den=dict(kidneys=PRIORS_FROZEN["kidneys"][0], liver=base.r_den["liver"], spleen=base.r_den["spleen"], salivary=F.PRIORS["salivary"][0]), f_sal=PRIORS_FROZEN["f_sal"][0])
    theta = main_theta.copy()
    theta[8] = np.log(median)
    return Co.forward_spread(base, theta, phi_list, rel_list, tau, nmol)


def metrics(rows):
    d = pd.DataFrame(rows).assign(tlv=TLV)
    high = d[HIGH]
    return dict(rho_kidneys_all=spearmanr(d.dose_kidneys, d.tlv)[0], rho_glands_all=spearmanr(d.dose_salivary, d.tlv)[0], rho_kidneys_high=spearmanr(high.dose_kidneys, high.tlv)[0], rho_glands_high=spearmanr(high.dose_salivary, high.tlv)[0], lesion_high=high.dose_lesions.mean(), lesion_all=d.dose_lesions.mean(), lesion_spread=float(np.log(d.dose_lesions.dropna()).std()), kidneys=d.dose_kidneys.mean(), glands=d.dose_salivary.mean(), sites_median=d.tumour_sites_nmol[HIGH].median(), sites_over_amount_high=float((d.tumour_sites_nmol[HIGH] > 100).mean()))


def grid_build():
    settings = [(m, t, a, False) for m in GRID_MEDIAN for t in GRID_TAU for a in GRID_NMOL] + [(181.0, 1.0, 100.0, True)]
    out = []
    with mp.get_context("fork").Pool(N_PROCESSES) as workers:
        for m, t, a, flat in settings:
            rows = workers.map(grid_job, [(i, m, t, a, flat) for i in range(len(BASES))], chunksize=12)
            out.append(dict(median=m, tau_scale=t, nmol=a, flat_density=flat, **metrics(rows)))
    return out


grid = pd.DataFrame(cached_v1("burden_grid", grid_build))
grid["closes"] = (grid.rho_kidneys_high.between(-0.73, -0.13)) & (grid.rho_glands_high.between(-0.71, -0.11)) & (np.abs(np.log(grid.lesion_high / LESION_HIGH_TARGET)) <= 0.3) & (np.abs(grid.lesion_spread - population_v1["spread_targets"]["dose_lesions"]) <= 0.2) & ~grid.flat_density
grid.to_csv(V1 / "burden_grid.csv", index=False)
print("Current setting and the control with one density for all patients")
display(grid[((grid["median"] == 181.0) & (grid.tau_scale == 1.0) & (grid.nmol == 100.0))].round(3).T)
print("Settings that close:", int(grid.closes.sum()), "of", int((~grid.flat_density).sum()))
display(grid[~grid.flat_density].groupby("nmol")[["rho_kidneys_high", "rho_glands_high", "rho_kidneys_all", "rho_glands_all", "lesion_high", "lesion_spread"]].agg(["min", "max"]).round(3).T)
display(grid[~grid.flat_density].sort_values("lesion_high").iloc[::6][["median", "tau_scale", "nmol", "rho_kidneys_high", "rho_glands_high", "lesion_high", "lesion_all", "lesion_spread", "kidneys", "glands", "sites_median", "sites_over_amount_high", "closes"]].round(3))
fig, axes = plt.subplots(1, 2, figsize=(9, 2.9), constrained_layout=True)
for nmol, color in zip(GRID_NMOL, COLORS):
    sub = grid[(grid.nmol == nmol) & (grid.tau_scale == 1.0) & ~grid.flat_density]
    axes[0].plot(sub["median"], sub.rho_kidneys_high, color=color, lw=1.6, label=f"{nmol:.0f} nmol")
    axes[1].plot(sub["median"], sub.lesion_high, color=color, lw=1.6)
axes[0].axhspan(-0.73, -0.13, color="#e6e6e3", zorder=0)
axes[1].axhline(LESION_HIGH_TARGET, color=MUTED, lw=0.8, ls=":")
axes[0].set(xscale="log", xlabel="Lesion median [R] (nmol/L)", ylabel="Spearman, kidney dose with TLV")
axes[1].set(xscale="log", xlabel="Lesion median [R] (nmol/L)", ylabel="Lesion Gy/GBq, TLV at least 100 mL")
axes[0].legend(frameon=False)
axes[0].set_title("Burden dependence of kidney dose", loc="left")
axes[1].set_title("Lesion dose at high burden", loc="left")
fig.savefig(V1 / "burden_grid.png", dpi=160)

## 8. Tumour burden: what was chosen, and what remains

**Diagnosis, from §7.** Too few tumour sites at therapy for ligand to be depleted: at TLV at least 100 mL, a median of 5.3 nmol against 100 nmol in the first cohort. The real-PET likelihood pulls the lesion draws far below the population median, the same pull as in nb18. The PET sink was about right (high over low TLV tertile: kidneys real 0.73, model 0.84; glands 0.84, 0.87), and the fit does not raise $[R_K]$ with burden: it falls (Spearman -0.20, -0.36 to -0.43 at TLV at least 100 mL).

**The 50 nmol closure was rejected, and why.** In the first grid only 2 of 54 settings met the criterion, both at a fixed 50 nmol. That was rejected because the range 50-200 nmol was our assumption, 50 lay below the only amounts read then (Kletting 2016: 74-302 nmol, I&T), and the closure was weak (glands -0.14 to -0.16). **The fixed 50 nmol stays rejected.** But the reason has changed since: **Wang 2025 measured clinical Pluvicto doses and gives 45-80 nmol, median 60 here**, so the assumed 100 nmol was the wrong one for Pluvicto, and values near 50 are within the measured range. The cohort uses the measurement, with its spread, not a chosen value.

**The choice.** The lesion median is not identifiable from the PET, so it was chosen on the targets: at the measured amount, the best of 27 settings is **median 800, SD × 1.15** (nb19 §17). That median is 2.9 prior SD above the five-pair start of 136. It is a calibration to published doses, not an estimate from these patients.

**The structural conflict.** A kidney correlation of -0.43 needs a median near 1600 (at the measured amount, -0.46), where lesion dose at high burden is 4.3 Gy/GBq, **2.5 times the target or more**. No setting fits both. Bringing it back through lesion release would need a half-life of about 35 h, against 69 h measured (order of magnitude, not simulated). The choice therefore favours lesion doses.

**The remaining gap, on the frozen cohort.**

| At TLV at least 100 mL | Target | First cohort | Frozen cohort |
|---|---|---|---|
| Lesion dose, Gy/GBq | 1.66 | 1.19 | **1.71** |
| Lesion dose spread (all) | 0.78 | 1.13 | **0.83** |
| Spearman kidney dose with TLV | -0.43 (±0.3) | -0.12 | **-0.07** |
| Spearman gland dose with TLV | -0.41 (±0.3) | -0.02 | **-0.06** |
| Tumour sites, median, nmol | | 5.3 of 100 | 11.5 of 60; 11% exceed the amount |

- **Lesions closed; organ correlations did not.** Over all phantoms Spearman is -0.04 and -0.08.
- **The grid promised more than the cohort gave, and this should be read as selection on noise.** After recalibration the grid gave -0.25 and -0.17 on 94 simulations from 47 phantoms. On the same phantoms with the same population, new fits under the higher median give -0.20 and -0.14, and the 470 virtual patients give -0.07 and -0.06. The grid's correlations carry an error of 0.1-0.15, and the setting was the best of 27 on them. The kidney interval of the criterion excludes the final value.
- **So a network trained on this cohort will hardly learn a dependence of organ dose on burden. Declared.** Lesion dose does fall with burden (-0.36 in rank; 3.7 Gy/GBq below 10 mL, 1.5 above 300 mL). Whole-body half-life rises with it (31 h below 10 mL, 41 h above 300 mL, target 39.9).

**Lesion mean over all patients** is 1.28 of its target, inside the tolerance (0.42): the price of matching high burden, with the case mix of autoPET.

**Note for the open finding of nb18.** The letter also cites 25 µg of PSMA-11 in the Locametz kit, while we assume 2 ± 0.5 nmol injected (Begum 2018, one centre). The mass per injected dose is not given and the LMU preparation may differ. This bears on "PET ligand amount", one of the three candidate causes; not pursued.

### Scope of the cohort

**autoPET is mostly low-burden staging patients, not mCRPC therapy candidates: median TLV 11.5 mL in the training split, 48% of phantoms at or below 10 mL, 26% at or above 100 mL. The virtual cohort does not represent the Pluvicto population.** Its calibrated means are over this case mix. Results on patients like those of VISION would be extrapolations.

### Network targets

**$A_0$ and $\lambda_w$ everywhere. $\lambda_u$ only where it is not degenerate.** Main arm, scenario A, training:

| Structure | Finite $\lambda_u$ | Early share of the TIA |
|---|---|---|
| Salivary glands | 36% | 2% |
| Lesions | 25% | 3% |
| Liver | 14% | 8% |
| Kidneys | 7% | 6% |
| Spleen | 6% | 12% |
| GI, rest of body, bladder, blood pool, spine | at most 0.1% | 26-75% |

So $\lambda_u$ is a target with a mask at most. `tiac_early_h` and `tiac_late_h` stay in the tables. **Cohort frozen. Training not started.**